# Calvin Site 1 • RLD search with a whole-vine mass constraint

Open from the project with the local research bundle available. Read saved results at both observation cutoffs and inspect the primary diameter evidence. Search ranks use measured core positions and depth increments. Each listed call regenerates a full-resolution vine.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
study = root / '.runtime/calvin-rld-search-20261119'
if not (study / 'FROZEN.json').is_file():
    raise FileNotFoundError(f'This review needs the local study bundle at {study}')
read = lambda path: json.loads(path.read_text())
display(Markdown((study / 'REPORT.md').read_text()))


## Standing-vine evidence and candidate priors

Compare the sampled basal envelope to the proximal mid-exposure diameters. The distance along each root to the field measurement is unknown.

In [ ]:
field = read(study / 'FIELD_EVIDENCE.json')
display({key:field[key] for key in ('count','angle_missing','mean_mm','sd_mm','min_mm','max_mm','top_two_area_fraction')})
display(pd.read_csv(study / 'primary_diameter_prior_checks.csv').groupby('stage').agg({'largest_possible_basal_mm':['min','max'],'chance_any_basal_at_least_47mm':['min','max']}))


## Essential parameters to optimize

Each per-order path uses `features.<feature>.settings.by_order.<order>`.

| Start with | Schema-4 settings | Read the response in |
|---|---|---|
| Primary diameter and taper | Order 1 `taper` → `base_cm`, `rate_per_cm`. The initial grid crosses basal distribution factors 1.00, 1.25, 1.50 and 1.75 with about 0.0035, 0.00657 and 0.00875 cm⁻¹. | Whole-vine mass, local daughter eligibility and RLD. |
| Length and branching | `lengths` distributions by order. In `branching`, adjust `daughter_probabilities` for parent orders 3–4 or `mean_spacing_cm` for parent orders 1–2 in the Poisson family. Consider separate order-2 and order-3 diameter changes later. | RLD magnitude and its distribution over depth and between cores. |
| Direction | In `curvature`, vary `row_alignment_per_cm` and `downward_drift_per_cm` on orders 1–3. Explore `downward_drift_per_cm` and `insertion_mean_deg` on orders 4–5. | Along-row spread, RLD depth profile and joint core-by-depth allocation. |

For the initial comparison, hold the nominal primary contour mean near 107.5 cm and its upper bound at 250 cm. Hold density at 0.45 g/cm³, the local parent gate at 0.093 cm and the daughter cap at 0.95. Compare measured and sampled crowns as named cases. Score RLD in the 0.93–2.50 mm band and use 0.50–2.50 mm for sensitivity. Check total dry mass against 5–7 kg for one vine, including its separately estimated underground crown/rootstock.

## Feasibility at the sampled core locations

An RLD ratio of 1 means the predicted mean matches the observed mean. Lower `cv_score` is better. `mass_in_band` gives the fraction of realizations within the mass band. Compare candidates with the same seed panels.

In [ ]:
ledger = pd.read_csv(study / 'all_attempts.csv')
display(ledger.groupby(['stage','status']).size().rename('attempts').reset_index())
ranking = pd.read_csv(study / 'development_frontier.csv')
display(ranking[['stage','case','seeds','cv_score','ratio','mass_mean_kg','mass_in_band','both_mean_guards']].head(20))
display(Image(filename=str(study / 'development_frontier.png'), width=720))


## Confirmation and replay

The finalist list is frozen using development data before the fresh confirmation panel. The three comparison cores have been inspected before.

In [ ]:
if (study / 'RESULT.json').is_file():
    result = read(study / 'RESULT.json')
    display(result)
    display(pd.read_csv(study / 'uncertainty.csv'))
    display(Image(filename=str(study / 'held_rld_profiles.png'), width=900))
    cfg = read(study / 'inputs' / (result['selected_in_development'] + '.schema4.private.json'))
    display(cfg['parameters'])
else:
    print('Final confirmation evidence is pending. See the execution ledger above.')


## Rebuild the review from the receipts

The following switch recomputes tables without creating new simulations. From the repository root, the bounded stage commands are `bash .runtime/calvin-rld-search-20261119/run.sh experiment.py` for replay and feasibility, and `bash .runtime/calvin-rld-search-20261119/run.sh adaptive.py` for the subsequent panels. The worker lock serializes execution.

In [ ]:
REBUILD_REVIEW = False
if REBUILD_REVIEW:
    import subprocess
    subprocess.run(['bash', str(study/'run.sh'), 'review.py'], cwd=root, check=True)
